# Signature-transform forecasting on the consumer-loans dataset

This notebook runs the point-forecasting pipeline on the original consumer-loans dataset. It imports reusable logic from `src/` and reproduces the deterministic multi-horizon experiments from the project.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from marketplace_signature_forecast.config import (
    ALL_HORIZONS,
    DATA_DIR,
    DEFAULT_DEPTH,
    DEFAULT_N_TARGET,
    DEFAULT_WINDOW_SIZE,
    END_DATE,
    FIGURES_DIR,
    FRED_API_KEY,
    N_VALIDATION,
    START_DATE,
    TARGET,
    ensure_directories,
)
from marketplace_signature_forecast.adaptive_weights import pick_gamma_by_neff, rolling_adaptive_weights
from marketplace_signature_forecast.baselines import compute_relative_error
from marketplace_signature_forecast.data_loading import build_data_dictionary, fetch_marketplace_series
from marketplace_signature_forecast.evaluation import (
    build_forecast_dataframe,
    compare_with_baselines,
    inverse_transform_forecasts,
    run_multi_horizon_experiment,
)
from marketplace_signature_forecast.modeling import rolling_forecast
from marketplace_signature_forecast.plotting import (
    plot_adaptive_weight_diagnostics,
    plot_correlation_matrix,
    plot_forecast_vs_actual,
    plot_input_grid,
    plot_target_split,
    plot_weights_for_origin,
)
from marketplace_signature_forecast.preprocessing import (
    build_model_dataset,
    prepare_standardized_arrays,
    resample_collection,
    resample_to_weekly,
    save_processed_bundle,
    train_validation_split,
)

ensure_directories()
np.random.seed(42)

## 1. Data download and preprocessing

In [ ]:
y_raw, x_raw = fetch_marketplace_series(FRED_API_KEY, START_DATE, END_DATE)
y_weekly = resample_to_weekly(y_raw)
x_weekly = resample_collection(x_raw)
full_data = build_model_dataset(y_weekly, x_weekly)
train_data, validation_data = train_validation_split(full_data, N_VALIDATION)
data_dictionary = build_data_dictionary()

save_processed_bundle(full_data, train_data, validation_data, data_dictionary, DATA_DIR)

print(full_data.shape)
full_data.head()

## 2. Exploratory analysis

In [ ]:
plot_target_split(train_data, validation_data, TARGET['name'], FIGURES_DIR / 'target_variable.png')
plot_input_grid(full_data, TARGET['name'], FIGURES_DIR / 'input_variables.png')
plot_correlation_matrix(full_data, FIGURES_DIR / 'correlation_matrix.png')

full_data.describe().round(2)

## 3. Standardized modeling arrays

In [ ]:
prepared = prepare_standardized_arrays(
    full_data=full_data,
    target_col=TARGET['name'],
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
)

X = prepared['X']
y = prepared['y']
y_raw_array = prepared['y_raw']
dates = prepared['dates']
y_scaler = prepared['y_scaler']
train_end_t = prepared['train_end']

WINDOW_SIZE = DEFAULT_WINDOW_SIZE
DEPTH = DEFAULT_DEPTH
N_TARGET = DEFAULT_N_TARGET
DELTA_T = 12


## 4. Adaptive-weight diagnostics

In [ ]:
min_t = WINDOW_SIZE + DELTA_T + 20
adaptive_results = rolling_adaptive_weights(
    X=X,
    y=y,
    start_t=min_t,
    end_t=train_end_t,
    delta_t=DELTA_T,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    gamma=None,
    n_target=N_TARGET,
    add_time=True,
)

plot_adaptive_weight_diagnostics(adaptive_results, dates, y, train_end_t, N_TARGET, TARGET['name'])
plot_weights_for_origin(adaptive_results, len(adaptive_results['t']) // 2, X, y, dates, DELTA_T, WINDOW_SIZE, DEPTH)

## 5. Single-horizon rolling forecast (12 weeks)

In [ ]:
val_start_t = len(y) - N_VALIDATION - DELTA_T
val_end_t = len(y) - DELTA_T - 1

forecast_results = rolling_forecast(
    X=X,
    y=y,
    start_t=val_start_t,
    end_t=val_end_t,
    delta_t=DELTA_T,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    lambda_lasso=None,
    gamma=None,
    n_target=N_TARGET,
    use_sig_y_only=True,
    add_time=True,
)

single_horizon_metrics = inverse_transform_forecasts(forecast_results, y_scaler)
single_horizon_df = build_forecast_dataframe(forecast_results, dates, DELTA_T, single_horizon_metrics)
single_horizon_df.to_csv(DATA_DIR / f'forecast_results_delta{DELTA_T}.csv', index=False)

print({k: v for k, v in single_horizon_metrics.items() if k in ['mae', 'rmse', 'mre', 'median_re']})
plot_forecast_vs_actual(single_horizon_df, DELTA_T)
single_horizon_df.head()

## 6. Multi-horizon signature experiment

In [ ]:
multi_horizon_results, signature_summary = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=DATA_DIR,
    lambda_lasso=None,
    gamma=None,
    use_sig_y_only=True,
    add_time=True,
)

signature_summary

## 7. Baseline comparison

In [ ]:
comparison_table = compare_with_baselines(
    y_raw=y_raw_array,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    signature_summary=signature_summary,
)

comparison_table.to_csv(DATA_DIR / 'table_a3_full_comparison.csv', index=False)
comparison_table

## 8. Optional synthetic sanity check for adaptive weights

In [ ]:
synthetic_rng = np.random.default_rng(42)
regime1 = np.cumsum(synthetic_rng.normal(loc=0.1, scale=0.5, size=30))
regime2 = regime1[-1] + np.cumsum(synthetic_rng.normal(loc=-0.3, scale=2.0, size=30))
regime3 = regime2[-1] + np.cumsum(synthetic_rng.normal(loc=0.1, scale=0.5, size=30))

synthetic_y = np.concatenate([regime1, regime2, regime3])
synthetic_X = np.zeros((90, 2))
gamma_grid = np.logspace(-3, 1, 25)
gamma_pick = pick_gamma_by_neff(gamma_grid, synthetic_X, synthetic_y, delta_t=5, window_size=10, depth=2, add_time=True, n_target=30)
gamma_pick